In [ ]:
import argparse
import math
import pysam
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

In [5]:
from collections import defaultdict, Counter

In [6]:
import sys
sys.path.append('../../test/py/')

from ReadPair import ReadPair

In [7]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "/data/ref/GRCh38_full_analysis_set_plus_decoy_hla.fa"
wgs_file  = working_path + "/data/WGS/ERR1726424.mkdup.sorted.bam"
wgbs_file = working_path + "/data/WGBS/ERR2359938.mkdup.sorted.bam"

In [8]:
read1_pos_qual_counter  = [Counter({})]*151
read2_pos_qual_counter  = [Counter({})]*151

read1_qual_trans_counter= Counter({})
read2_qual_trans_counter= Counter({})

# 43 scores 
##https://www.drive5.com/usearch/manual/quality_score.html#:~:text=With%20a%20typical%20range%20from,from%20ASCII%2066%20to%2073.
read1_qual_err_dict = {i:Counter({}) for i in range(43)}
read2_qual_err_dict = {i:Counter({}) for i in range(43)}

mismatch_rec_dict = {}
pos_base_dict = defaultdict(Counter)

In [ ]:
def align_filter(align, min_mapping_quality = 30):
    is_mapped  = (not align.is_unmapped)
    is_primary = not (align.is_secondary or align.is_supplementary) ## secondary and supplementary is removed in read_pair_generator
    is_qualified = (not align.is_qcfail) and align.mapping_quality >= min_mapping_quality
    not_duplicate = not align.is_duplicate
    is_fully_align = all(op in (0, 7, 8) for op, length in align.cigar)
    return is_mapped and is_primary and is_qualified and not_duplicate and is_fully_align
       

def collect_qual_trans(bam_file, contig=None, start=None, stop=None):    
    # assumed used directional library
    for read1, read2 in read_pair_generator(bam_file, contig=contig, start=start, stop=stop):
        # record quality along the read, and quality transition matrix
        read1_5to3 = 1 if read1.get_tag('YS') == "W_C2T" else 0
        read1_qual = read1.get_forward_qualities() if read1_5to3 else np.flip(read1.get_forward_qualities())
        read2_qual = np.flip(read2.get_forward_qualities()) if read1_5to3 else read2.get_forward_qualities()

        for ix, base_q in enumerate(read1_qual):
            read1_pos_qual_counter[ix].update([base_q])
            if ix:
                read1_qual_trans_counter.update([(read1_qual[ix-1], base_q)])

        for ix, base_q in enumerate(read2_qual):
            read2_pos_qual_counter[ix].update([base_q])
            if ix:
                read2_qual_trans_counter.update([(read2_qual[ix-1], base_q)])

def collect_qual_err(bam_file, contig=None, start=None, stop=None):
    # get sequencing error for each quality
    for read1, read2 in read_pair_generator(bam_file, contig=contig, start=start, stop=stop):
        if align_filter(read1) and align_filter(read2):# no indel or failed or duplicate        
            s1 = read1.reference_start # coordinate is half open [start, end)
            e1 = read1.reference_end
            s2 = read2.reference_start
            e2 = read2.reference_end

            if (s2-e1<0 and s1-e2<0) or (s1-e2<0 and s2-e1<0): # has overlap
                read1_pos = np.array([site[1] for site in read1.aligned_pairs])
                read2_pos = np.array([site[1] for site in read2.aligned_pairs])
                overlap_pos = np.intersect1d(read1_pos, read2_pos)
                read1_qpos_l = np.squeeze(np.where(read1_pos == np.min(overlap_pos)))
                read2_qpos_l = np.squeeze(np.where(read2_pos == np.min(overlap_pos)))
                
                read1_5to3 = 1 if read1.get_tag('YS') == "W_C2T" else 0
                read1_qual = read1.get_forward_qualities()
                read2_qual = read2.get_forward_qualities()

                for ix, idx in enumerate(range(read1_qpos_l, read1_qpos_l+overlap_pos.size)):
                    ref_pos = read1.aligned_pairs[idx][1]
                    read1_base = read1.seq[idx]
                    read1_base_qual = read1_qual[idx]
                    read2_base = read2.seq[read2_qpos_l + ix]
                    read2_base_qual = read2_qual[read2_qpos_l + ix]
                    if 'N' in [read1_base, read2_base]: # interesting, read can have N
                        continue

                    pos_base_dict[ref_pos][read1_base] += 1
                    pos_base_dict[ref_pos][read2_base] += 1

                    if read1_base == read2_base:
                        read1_qual_err_dict[read1_base_qual].update([(read2_base, read1_base)])
                        read2_qual_err_dict[read2_base_qual].update([(read1_base, read2_base)])
                    else: # hold, determine which one is correct later
                        if ref_pos not in mismatch_rec_dict:
                            mismatch_rec_dict[ref_pos] = Counter()
                        mismatch_rec_dict[ref_pos].update([(read1_5to3, read1_base, read1_base_qual, read2_base, read2_base_qual)])

def update_qual_err(mismatch_rec_dict, pos_base_dict):
    for pos in mismatch_rec_dict.keys():
        mismatch_rec_counter = mismatch_rec_dict[pos]
        common_base, _ = pos_base_dict[pos].most_common()[0]
        common_base_c = {'A':'T', 'C':'G', 'G':'C', 'T':'A'}[common_base]
        for read1_5to3, read1_base, read1_base_qual, read2_base, read2_base_qual in mismatch_rec_counter:
            if read1_5to3:
                read1_qual_err_dict[read1_base_qual].update([(common_base, read1_base)])
                read2_qual_err_dict[read2_base_qual].update([(common_base_c, {'A':'T', 'C':'G', 'G':'C', 'T':'A'}[read2_base])])
            else:
                read1_qual_err_dict[read1_base_qual].update([(common_base_c, {'A':'T', 'C':'G', 'G':'C', 'T':'A'}[read1_base])])
                read2_qual_err_dict[read2_base_qual].update([(common_base, read2_base)])


In [ ]:
collect_qual_trans(wgbs_file, contig='chr21', start = 5220063, stop= 6220884)

In [ ]:
collect_qual_err(wgbs_file, contig='chr21', start = 5220063, stop= 6220884)

In [ ]:
update_qual_err(mismatch_rec_dict, pos_base_dict)

In [ ]:
mismatch_rec_dict

In [ ]:
for item in mismatch_rec_dict[5220579].keys():
    print(item, mismatch_rec_dict[5220579][item])